# Train a small GPT, upcycle its FFNs into MoE, and continue training

Self-contained source snapshot. Select a GPU runtime, then Run all. No GitHub repository publication or file upload is required. The baseline is trained from scratch on Tiny Shakespeare. Completed Colab results and evidence are documented in README.md.

The comparison matches target-token exposure, not FLOPs. Do not infer MoE superiority merely from a small loss difference. Save the final downloadable archive before the runtime disconnects.

In [ ]:
import sys, subprocess, importlib.util
if importlib.util.find_spec("matplotlib") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib>=3.8,<4"], check=True)
import torch
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "Select a GPU runtime before running this notebook."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)


In [ ]:
from pathlib import Path
import os, json
root = Path("/content/MoE_Training")
root.mkdir(parents=True, exist_ok=True)
os.chdir(root)
payload = {'src/gpt.py': '"""Small causal decoder-only Transformer and FFN-upcycled sparse MoE."""\nimport copy\nimport math\nfrom dataclasses import dataclass\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n@dataclass\nclass GPTConfig:\n    vocab_size: int = 65\n    context: int = 128\n    layers: int = 4\n    width: int = 128\n    heads: int = 4\n    intermediate: int = 256\n    experts: int = 4\n    top_k: int = 2\n\nclass SwiGLU(nn.Module):\n    def __init__(self, width, intermediate):\n        super().__init__()\n        self.gate = nn.Linear(width, intermediate, bias=False)\n        self.up = nn.Linear(width, intermediate, bias=False)\n        self.down = nn.Linear(intermediate, width, bias=False)\n\n    def forward(self, x):\n        return self.down(F.silu(self.gate(x)) * self.up(x))\n\nclass ExpertFFN(nn.Module):\n    def __init__(self, dense_ffn, width, experts, top_k):\n        super().__init__()\n        if not 1 <= top_k <= experts:\n            raise ValueError(\'Require 1 <= top_k <= experts\')\n        self.top_k = top_k\n        self.experts = nn.ModuleList([copy.deepcopy(dense_ffn) for _ in range(experts)])\n        self.router = nn.Linear(width, experts, bias=False)\n        nn.init.normal_(self.router.weight, std=.02)\n\n    def forward(self, hidden):\n        shape = hidden.shape\n        x = hidden.reshape(-1, shape[-1])\n        # FP32 for ordinary training/autocast, FP64 for independent CPU\n        # reference validation. Do not downcast a double reference gate.\n        gate_dtype = torch.float64 if x.dtype == torch.float64 else torch.float32\n        logits = F.linear(x.to(gate_dtype), self.router.weight.to(gate_dtype))\n        probs = logits.softmax(-1)\n        values, ids = logits.topk(self.top_k, dim=-1)\n        weights = values.softmax(-1).to(x.dtype)\n        output = torch.zeros_like(x)\n        for i, expert in enumerate(self.experts):\n            rows, slots = torch.where(ids == i)\n            if rows.numel():\n                contribution = expert(x[rows]) * weights[rows, slots, None]\n                output = output.index_add(0, rows, contribution)\n        counts = torch.bincount(ids.flatten(), minlength=len(self.experts)).float()\n        fraction = counts / ids.numel()\n        balance = len(self.experts) * (fraction.detach() * probs.mean(0)).sum()\n        entropy = -(probs * probs.clamp_min(1e-9).log()).sum(-1).mean()\n        return output.reshape(shape), {\'balance\': balance, \'counts\': counts, \'entropy\': entropy}\n\nclass Attention(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        if cfg.width % cfg.heads:\n            raise ValueError(\'width must be divisible by heads\')\n        self.heads = cfg.heads\n        self.qkv = nn.Linear(cfg.width, 3*cfg.width, bias=False)\n        self.proj = nn.Linear(cfg.width, cfg.width, bias=False)\n\n    def forward(self, x):\n        b,t,d = x.shape\n        q,k,v = self.qkv(x).chunk(3, dim=-1)\n        q,k,v = [a.reshape(b,t,self.heads,d//self.heads).transpose(1,2) for a in (q,k,v)]\n        y = F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.)\n        return self.proj(y.transpose(1,2).contiguous().reshape(b,t,d))\n\nclass Block(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.norm_attn = nn.LayerNorm(cfg.width)\n        self.attn = Attention(cfg)\n        self.norm_ffn = nn.LayerNorm(cfg.width)\n        self.ffn = SwiGLU(cfg.width, cfg.intermediate)\n\n    def forward(self, x):\n        x = x + self.attn(self.norm_attn(x))\n        if isinstance(self.ffn, ExpertFFN):\n            y, aux = self.ffn(self.norm_ffn(x))\n        else:\n            y, aux = self.ffn(self.norm_ffn(x)), None\n        return x+y, aux\n\nclass TinyGPT(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        self.token_embedding = nn.Embedding(cfg.vocab_size,cfg.width)\n        self.position_embedding = nn.Embedding(cfg.context,cfg.width)\n        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.layers)])\n        self.norm = nn.LayerNorm(cfg.width)\n        self.head = nn.Linear(cfg.width,cfg.vocab_size,bias=False)\n        self.head.weight = self.token_embedding.weight\n        self.apply(self._initialize)\n        for name,p in self.named_parameters():\n            if name.endswith((\'attn.proj.weight\',\'ffn.down.weight\')):\n                nn.init.normal_(p, std=.02/math.sqrt(2*cfg.layers))\n\n    @staticmethod\n    def _initialize(module):\n        if isinstance(module,(nn.Linear,nn.Embedding)):\n            nn.init.normal_(module.weight,std=.02)\n            if isinstance(module,nn.Linear) and module.bias is not None:\n                nn.init.zeros_(module.bias)\n\n    def forward(self, tokens):\n        if tokens.shape[1] > self.cfg.context:\n            raise ValueError(\'Sequence exceeds configured context\')\n        x = self.token_embedding(tokens) + self.position_embedding(torch.arange(tokens.shape[1],device=tokens.device))\n        aux=[]\n        for block in self.blocks:\n            x,a = block(x)\n            if a is not None:\n                aux.append(a)\n        return self.head(self.norm(x)), aux\n\n    def upcycle(self):\n        result = copy.deepcopy(self)\n        if any(isinstance(b.ffn,ExpertFFN) for b in result.blocks):\n            raise ValueError(\'Model is already an MoE\')\n        for block in result.blocks:\n            block.ffn = ExpertFFN(block.ffn,self.cfg.width,self.cfg.experts,self.cfg.top_k).to(self.head.weight.device)\n        return result\n\n    def parameter_counts(self):\n        total=sum(p.numel() for p in self.parameters())\n        inactive=0\n        for block in self.blocks:\n            if isinstance(block.ffn,ExpertFFN):\n                each=sum(p.numel() for p in block.ffn.experts[0].parameters())\n                inactive += (len(block.ffn.experts)-block.ffn.top_k)*each\n        return {\'total\':total, \'active_parameter_convention\':total-inactive,\n                \'convention\':\'All nonexpert parameters, including full tied embedding table, plus selected expert parameters. Not a FLOP count.\'}\n', 'src/train_gpt.py': '"""Dense GPT -> FFN-upcycled MoE, with matched-token dense continuation."""\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport platform\nimport subprocess\nimport time\nfrom dataclasses import asdict\nfrom pathlib import Path\nimport torch\nfrom torch.nn import functional as F\nfrom .gpt import GPTConfig,TinyGPT,ExpertFFN\nfrom .train import load_data\nfrom .conversion_checks import verify_model_conversion\n\n\ndef sample(data,gen,n,context):\n    starts=torch.randint(len(data)-context,(n,),generator=gen)\n    sequence=data[starts[:,None]+torch.arange(context+1)]\n    return sequence[:,:-1],sequence[:,1:]\n\n\ndef step(model,opt,batch,device,cfg):\n    model.train()\n    opt.zero_grad(set_to_none=True)\n    x,y=(t.to(device) for t in batch)\n    logits,aux=model(x)\n    ce=F.cross_entropy(logits.reshape(-1,logits.shape[-1]),y.reshape(-1))\n    balance=torch.stack([a[\'balance\'] for a in aux]).mean() if aux else ce.new_tensor(0.)\n    objective=ce+cfg[\'balance_coefficient\']*balance\n    objective.backward()\n    norm=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg[\'gradient_clip\'])\n    if not torch.isfinite(norm) or not torch.isfinite(objective):\n        raise RuntimeError(\'Non-finite gradients/objective\')\n    router_grad=[b.ffn.router.weight.grad.norm().item() for b in model.blocks if isinstance(b.ffn,ExpertFFN)]\n    opt.step()\n    return {\'batch_task_ce\':ce.item(),\'balance_loss\':balance.item(),\'objective\':objective.item(),\n            \'gradient_norm_before_clip\':norm.item(),\'router_gradient_norm\':router_grad}\n\n\n@torch.no_grad()\ndef evaluate(model,batch,device):\n    model.eval()\n    loss=0.\n    loads=[]\n    entropy=[]\n    for x,y in zip(batch[0].split(8),batch[1].split(8)):\n        logits,aux=model(x.to(device))\n        loss+=F.cross_entropy(logits.reshape(-1,logits.shape[-1]),y.to(device).reshape(-1),reduction=\'sum\').item()\n        if aux:\n            counts=torch.stack([a[\'counts\'] for a in aux]).cpu()\n            if not loads:\n                loads=[torch.zeros_like(counts)]\n                entropy=[torch.zeros(len(aux))]\n            loads[0]+=counts\n            entropy[0]+=torch.tensor([a[\'entropy\'].item() for a in aux])*y.numel()\n    result={\'task_ce\':loss/batch[1].numel()}\n    if loads:\n        fractions=loads[0]/loads[0].sum(-1,keepdim=True)\n        result.update(expert_fractions=fractions.tolist(),\n                      router_entropy=(entropy[0]/batch[1].numel()).tolist(),\n                      expert_load_cv=(fractions.std(-1,unbiased=False)/fractions.mean(-1)).tolist())\n    return result\n\n\ndef phase(model,cfg,data,evals,seed,name,steps,lr,offset,device,checkpoint_dir,log_path):\n    opt=torch.optim.AdamW(model.parameters(),lr=lr,betas=(.9,.95),weight_decay=cfg[\'weight_decay\'])\n    gen=torch.Generator().manual_seed(seed+(10000 if name==\'dense_pretrain\' else 20000))\n    start=time.perf_counter()\n    records=[]\n    last={}\n    for i in range(steps+1):\n        if i%cfg[\'eval_every\']==0 or i==steps:\n            train=evaluate(model,evals[\'train\'],device)\n            validation=evaluate(model,evals[\'validation\'],device)\n            record={\'seed\':seed,\'phase\':name,\'step\':i,\'global_step\':offset+i,\n                    \'targets_seen\':(offset+i)*cfg[\'batch_size\']*cfg[\'model\'][\'context\'],\n                    \'train_ce\':train[\'task_ce\'],\'validation_ce\':validation[\'task_ce\'],\n                    \'elapsed_seconds_including_eval\':time.perf_counter()-start,**last}\n            for k in (\'expert_fractions\',\'router_entropy\',\'expert_load_cv\'):\n                if k in validation:\n                    record[k]=validation[k]\n            records.append(record)\n            with log_path.open(\'a\') as f:\n                f.write(json.dumps(record)+\'\\n\')\n            print(json.dumps(record),flush=True)\n        if i==steps:\n            break\n        last=step(model,opt,sample(data,gen,cfg[\'batch_size\'],cfg[\'model\'][\'context\']),device,cfg)\n    if device.startswith(\'cuda\'):\n        torch.cuda.synchronize()\n    elapsed=time.perf_counter()-start\n    saved={\'model\':model.state_dict(),\'optimizer\':opt.state_dict(),\'sampler_rng\':gen.get_state(),\n           \'torch_rng\':torch.get_rng_state(),\'cuda_rng\':torch.cuda.get_rng_state_all() if device.startswith(\'cuda\') else [],\n           \'config\':cfg,\'seed\':seed,\'phase\':name,\'step\':steps}\n    path=checkpoint_dir/f\'{name}.pt\'\n    torch.save(saved,path)\n    # Check on-disk model/Adam/sampler restoration with an identical next update.\n    loaded=torch.load(path,map_location=device,weights_only=True)\n    restored=copy.deepcopy(model)\n    restored.load_state_dict(loaded[\'model\'])\n    restored_opt=torch.optim.AdamW(restored.parameters(),lr=lr,betas=(.9,.95),weight_decay=cfg[\'weight_decay\'])\n    restored_opt.load_state_dict(loaded[\'optimizer\'])\n    restored_gen=torch.Generator()\n    restored_gen.set_state(loaded[\'sampler_rng\'].cpu())\n    batch=sample(data,gen,cfg[\'batch_size\'],cfg[\'model\'][\'context\'])\n    restored_batch=sample(data,restored_gen,cfg[\'batch_size\'],cfg[\'model\'][\'context\'])\n    assert all(torch.equal(a,b) for a,b in zip(batch,restored_batch))\n    reference=copy.deepcopy(model)\n    ref_opt=torch.optim.AdamW(reference.parameters(),lr=lr,betas=(.9,.95),weight_decay=cfg[\'weight_decay\'])\n    ref_opt.load_state_dict(copy.deepcopy(opt.state_dict()))\n    step(reference,ref_opt,batch,device,cfg)\n    step(restored,restored_opt,restored_batch,device,cfg)\n    error=max((a-b).abs().max().item() for a,b in zip(reference.parameters(),restored.parameters()))\n    assert error<1e-6,error\n    return records,{\'wall_seconds_including_eval\':elapsed,\n                    \'targets_per_second_including_eval\':steps*cfg[\'batch_size\']*cfg[\'model\'][\'context\']/elapsed,\n                    \'resume_next_update_parameter_error\':error}\n\n\n@torch.no_grad()\ndef generate(model,vocabulary,prompt,device,seed,length=200):\n    gen=torch.Generator(device=device).manual_seed(seed)\n    mapping={c:i for i,c in enumerate(vocabulary)}\n    tokens=torch.tensor([[mapping[c] for c in prompt]],device=device)\n    model.eval()\n    for _ in range(length):\n        logits,_=model(tokens[:,-model.cfg.context:])\n        next_token=torch.multinomial((logits[:,-1]/.8).softmax(-1),1,generator=gen)\n        tokens=torch.cat((tokens,next_token),dim=1)\n    return \'\'.join(vocabulary[i] for i in tokens[0].tolist())\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--config\',default=\'configs/gpt_cpu.json\')\n    p.add_argument(\'--data\',default=\'data/tinyshakespeare.txt\')\n    p.add_argument(\'--device\',default=\'cpu\')\n    p.add_argument(\'--output\',default=\'results/gpt-cpu-pilot\')\n    p.add_argument(\'--threads\',type=int,default=2)\n    p.add_argument(\'--resume-dense-run\',help=\'Prior result directory with completed dense phase and checkpoints/<run-name>/<seed>/dense_pretrain.pt\')\n    a=p.parse_args()\n    torch.set_num_threads(a.threads)\n    cfg=json.loads(Path(a.config).read_text())\n    out=Path(a.output)\n    out.mkdir(parents=True,exist_ok=False)\n    (out/\'config.json\').write_text(json.dumps(cfg,indent=2))\n    splits,manifest=load_data(Path(a.data),cfg[\'model\'][\'context\'])\n    manifest[\'objective\']=\'Causal next-character prediction at every position; input/target shifted by one.\'\n    (out/\'data_manifest.json\').write_text(json.dumps(manifest,indent=2))\n    evals={k:sample(v,torch.Generator().manual_seed(9000+i),cfg[\'eval_sequences\'],cfg[\'model\'][\'context\']) for i,(k,v) in enumerate(splits.items())}\n    try:\n        commit=subprocess.check_output([\'git\',\'rev-parse\',\'HEAD\'],stderr=subprocess.DEVNULL,text=True).strip()\n    except subprocess.CalledProcessError:\n        commit=None\n    hashes={str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(\'src\').glob(\'*.py\')}\n    env={\'python\':platform.python_version(),\'torch\':str(torch.__version__),\'device\':a.device,\'threads\':a.threads,\n         \'gpu_name\':torch.cuda.get_device_name() if a.device.startswith(\'cuda\') else None,\n         \'git_commit\':commit,\'source_sha256\':hashes,\'precision\':\'FP32; no autocast/dropout\'}\n    (out/\'environment.json\').write_text(json.dumps(env,indent=2))\n    summaries=[]\n    samples={}\n    for seed in cfg[\'seeds\']:\n        torch.manual_seed(seed)\n        if a.device.startswith(\'cuda\'):\n            torch.cuda.reset_peak_memory_stats()\n        checkpoint_dir=Path(\'checkpoints\')/out.name/str(seed)\n        checkpoint_dir.mkdir(parents=True,exist_ok=True)\n        model=TinyGPT(GPTConfig(vocab_size=len(manifest[\'vocabulary\']),**cfg[\'model\'])).to(a.device)\n        if a.resume_dense_run:\n            source=Path(a.resume_dense_run)\n            source_manifest=json.loads((source/\'data_manifest.json\').read_text())\n            if source_manifest[\'sha256\']!=manifest[\'sha256\']:\n                raise ValueError(\'Resume dataset checksum mismatch\')\n            path=Path(\'checkpoints\')/source.name/str(seed)/\'dense_pretrain.pt\'\n            loaded=torch.load(path,map_location=a.device,weights_only=True)\n            if loaded[\'seed\']!=seed or loaded[\'phase\']!=\'dense_pretrain\' or loaded[\'step\']!=cfg[\'dense_steps\']:\n                raise ValueError(\'Resume checkpoint seed/phase/step mismatch\')\n            for key in (\'model\',\'dense_steps\',\'dense_lr\',\'batch_size\',\'weight_decay\',\'gradient_clip\',\'eval_sequences\'):\n                if loaded[\'config\'][key]!=cfg[key]:\n                    raise ValueError(f\'Resume configuration mismatch: {key}\')\n            model.load_state_dict(loaded[\'model\'])\n            torch.set_rng_state(loaded[\'torch_rng\'].cpu())\n            if a.device.startswith(\'cuda\') and loaded[\'cuda_rng\']:\n                torch.cuda.set_rng_state_all([r.cpu() for r in loaded[\'cuda_rng\']])\n            dense_records=[json.loads(line) for line in (source/\'metrics.jsonl\').read_text().splitlines()]\n            dense_records=[r for r in dense_records if r[\'seed\']==seed and r[\'phase\']==\'dense_pretrain\']\n            if not dense_records or dense_records[0][\'step\']!=0 or dense_records[-1][\'step\']!=cfg[\'dense_steps\']:\n                raise ValueError(\'Resume requires complete dense metrics\')\n            with (out/\'metrics.jsonl\').open(\'a\') as f:\n                for r in dense_records:f.write(json.dumps(r)+\'\\n\')\n            dense_timing={\'wall_seconds_including_eval\':dense_records[-1][\'elapsed_seconds_including_eval\'],\n                          \'resumed_from\':str(source),\'checkpoint_sha256\':hashlib.sha256(path.read_bytes()).hexdigest(),\n                          \'resume_next_update_parameter_error\':None,\n                          \'note\':\'Dense timing/metrics inherited; continuation optimizer resets as before.\'}\n            torch.save(loaded,checkpoint_dir/\'dense_pretrain.pt\')\n            print(f\'Restored dense checkpoint: {path}; skipping pretraining\',flush=True)\n        else:\n            dense_records,dense_timing=phase(model,cfg,splits[\'train\'],evals,seed,\'dense_pretrain\',cfg[\'dense_steps\'],cfg[\'dense_lr\'],0,a.device,checkpoint_dir,out/\'metrics.jsonl\')\n        model.eval()\n        moe=model.upcycle().eval()\n        # Exact parameter-copy checks catch structural errors independently of\n        # floating-point kernel differences in sparse subset execution.\n        for dense_block,moe_block in zip(model.blocks,moe.blocks):\n            for expert in moe_block.ffn.experts:\n                for original,copied in zip(dense_block.ffn.parameters(),expert.parameters()):\n                    if not torch.equal(original,copied):\n                        raise RuntimeError(\'Expert weight copy mismatch\')\n        with torch.no_grad():\n            probe,targets=(t.to(a.device) for t in evals[\'validation\'])\n            conversion=verify_model_conversion(model,moe,probe,targets)\n        conversion_error=conversion[\'max_absolute_logit_error\']\n        (out/f\'conversion-{seed}.json\').write_text(json.dumps(conversion,indent=2))\n        print(\'Conversion diagnostics:\',json.dumps(conversion),flush=True)\n        if not conversion[\'passed\']:\n            raise RuntimeError(f\'Conversion check failed; see {out}/conversion-{seed}.json: {conversion}\')\n        torch.save({\'model\':moe.state_dict(),\'config\':cfg,\'seed\':seed},checkpoint_dir/\'moe_converted.pt\')\n        original_experts={k:v.detach().clone() for k,v in moe.state_dict().items() if \'.experts.\' in k}\n        branches={}\n        for name,branch in [(\'dense_control\',copy.deepcopy(model)),(\'moe_continue\',moe)]:\n            records,timing=phase(branch,cfg,splits[\'train\'],evals,seed,name,cfg[\'continuation_steps\'],cfg[\'continuation_lr\'],cfg[\'dense_steps\'],a.device,checkpoint_dir,out/\'metrics.jsonl\')\n            branches[name]={\'start\':records[0],\'final\':records[-1],\'test_ce\':evaluate(branch,evals[\'test\'],a.device)[\'task_ce\'],**timing}\n            samples[f\'{seed}/{name}\']=generate(branch,manifest[\'vocabulary\'],\'First Citizen:\\n\',a.device,1234)\n        deltas={k:(v-original_experts[k]).norm().item() for k,v in moe.state_dict().items() if k in original_experts}\n        all_updated=all(v>0 for v in deltas.values())\n        acceptance={\'dense_train_drops\':dense_records[-1][\'train_ce\']<dense_records[0][\'train_ce\'],\n                    \'dense_validation_drops\':dense_records[-1][\'validation_ce\']<dense_records[0][\'validation_ce\'],\n                    \'moe_train_drops\':branches[\'moe_continue\'][\'final\'][\'train_ce\']<dense_records[-1][\'train_ce\'],\n                    \'moe_validation_drops\':branches[\'moe_continue\'][\'final\'][\'validation_ce\']<dense_records[-1][\'validation_ce\'],\n                    \'all_expert_matrices_updated\':all_updated,\'conversion_preserved\':conversion[\'passed\']}\n        summary={\'seed\':seed,\'model_config\':asdict(model.cfg),\'dense_parameters\':model.parameter_counts(),\n                 \'moe_parameters\':moe.parameter_counts(),\'conversion_max_logit_error\':conversion_error,\'conversion_diagnostics\':conversion,\n                 \'initial\':dense_records[0],\'dense_end\':dense_records[-1],\'dense_timing\':dense_timing,\n                 \'branches\':branches,\'expert_matrix_changes_l2\':deltas,\'acceptance\':acceptance,\n                 \'peak_cuda_memory_allocated_bytes\':torch.cuda.max_memory_allocated() if a.device.startswith(\'cuda\') else None}\n        summaries.append(summary)\n        (out/\'summary.json\').write_text(json.dumps(summaries,indent=2))\n        (out/\'generation_samples.json\').write_text(json.dumps(samples,indent=2))\n    print(\'Acceptance:\',json.dumps([s[\'acceptance\'] for s in summaries]),flush=True)\n    if not all(all(s[\'acceptance\'].values()) for s in summaries):\n        raise SystemExit(\'Acceptance failure; preserve the run and investigate, without overwriting it.\')\n\nif __name__==\'__main__\':\n    main()\n', 'src/train.py': '"""Run from repository root: python -m src.train --device cpu"""\nimport argparse\nimport copy\nimport csv\nimport hashlib\nimport json\nimport platform\nimport subprocess\nimport time\nfrom pathlib import Path\nimport torch\nfrom torch.nn import functional as F\nfrom .models import LinearLM, SparseMoE, parameters\n\n\ndef windows(data, starts, context):\n    return data[starts[:, None] + torch.arange(context)], data[starts + context]\n\n\ndef load_data(path, context):\n    raw = path.read_bytes()\n    text = raw.decode(\'utf-8\')\n    # Split raw text first: no input or target can cross a split boundary.\n    a, b = int(.8 * len(text)), int(.9 * len(text))\n    vocab = sorted(set(text[:a]))\n    mapping = {c: i for i, c in enumerate(vocab)}\n    # This corpus has the same character support in all splits; fail if changed.\n    splits = {k: torch.tensor([mapping[c] for c in t]) for k, t in\n              [(\'train\', text[:a]), (\'validation\', text[a:b]), (\'test\', text[b:])]}\n    manifest = {\'source\': \'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt\',\n                \'sha256\': hashlib.sha256(raw).hexdigest(), \'bytes\': len(raw),\n                \'characters\': len(text), \'vocabulary\': vocab,\n                \'split_characters\': {k: len(v) for k, v in splits.items()},\n                \'split_method\': \'contiguous 80/10/10; split before constructing windows\',\n                \'context\': context}\n    return splits, manifest\n\n\ndef fixed_batch(data, context, n, seed):\n    gen = torch.Generator().manual_seed(seed)\n    starts = torch.randint(len(data) - context, (n,), generator=gen)\n    return windows(data, starts, context)\n\n\n@torch.no_grad()\ndef evaluate(model, batch, device):\n    model.eval()\n    losses, counts, entropies = [], [], []\n    for x, y in zip(batch[0].split(512), batch[1].split(512)):\n        logits, aux = model(x.to(device))\n        losses.append(F.cross_entropy(logits, y.to(device), reduction=\'sum\').item())\n        if aux is not None:\n            counts.append(aux[\'counts\'].cpu())\n            entropies.append((aux[\'entropy\'].item(), len(y)))\n    out = {\'ce\': sum(losses) / len(batch[1])}\n    if counts:\n        load = torch.stack(counts).sum(0)\n        out[\'expert_fraction\'] = (load / load.sum()).tolist()\n        out[\'router_entropy\'] = sum(e*n for e,n in entropies) / len(batch[1])\n    return out\n\n\ndef update(model, optimizer, batch, device, cfg):\n    model.train()\n    optimizer.zero_grad(set_to_none=True)\n    logits, aux = model(batch[0].to(device))\n    ce = F.cross_entropy(logits, batch[1].to(device))\n    objective = ce + (cfg[\'balance_coefficient\'] * aux[\'balance\'] if aux else 0)\n    objective.backward()\n    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg[\'gradient_clip\'])\n    if not torch.isfinite(norm) or not torch.isfinite(objective):\n        raise RuntimeError(\'Non-finite gradient or loss\')\n    optimizer.step()\n    return {\'batch_ce\': ce.item(), \'objective\': objective.item(), \'grad_norm\': norm.item()}\n\n\ndef run_phase(model, cfg, splits, evals, device, seed, phase, steps, lr, offset, folder):\n    # Both continuation branches deliberately reset AdamW at conversion.\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=cfg[\'weight_decay\'])\n    gen = torch.Generator().manual_seed(seed + (10000 if phase == \'dense_pretrain\' else 20000))\n    records = []\n    start = time.perf_counter()\n    last = {}\n    for step in range(steps + 1):\n        if step % cfg[\'eval_every\'] == 0 or step == steps:\n            tr = evaluate(model, evals[\'train\'], device)\n            va = evaluate(model, evals[\'validation\'], device)\n            rec = {\'seed\': seed, \'phase\': phase, \'step\': step, \'global_step\': offset + step,\n                   \'targets_seen\': (offset + step) * cfg[\'batch_size\'],\n                   \'train_ce\': tr[\'ce\'], \'validation_ce\': va[\'ce\'],\n                   \'elapsed_seconds_including_eval\': time.perf_counter() - start,\n                   **last}\n            if \'expert_fraction\' in va:\n                rec.update(expert_fraction=va[\'expert_fraction\'], router_entropy=va[\'router_entropy\'])\n            records.append(rec)\n            print(json.dumps(rec), flush=True)\n        if step == steps:\n            break\n        indices = torch.randint(len(splits[\'train\']) - cfg[\'context\'], (cfg[\'batch_size\'],), generator=gen)\n        last = update(model, opt, windows(splits[\'train\'], indices, cfg[\'context\']), device, cfg)\n    if device.startswith(\'cuda\'):\n        torch.cuda.synchronize()\n    elapsed = time.perf_counter() - start\n    checkpoint = {\'model\': model.state_dict(), \'optimizer\': opt.state_dict(),\n                  \'sampler_rng\': gen.get_state(), \'torch_rng\': torch.get_rng_state(),\n                  \'cuda_rng\': torch.cuda.get_rng_state_all() if device.startswith(\'cuda\') else [],\n                  \'phase\': phase, \'step\': steps, \'seed\': seed, \'config\': cfg}\n    checkpoint_path = folder / f\'{phase}.pt\'\n    torch.save(checkpoint, checkpoint_path)\n    # Verify actual disk reload, optimizer restoration, sampler restoration and next update.\n    loaded = torch.load(checkpoint_path, map_location=device, weights_only=True)\n    clone = copy.deepcopy(model)\n    clone.load_state_dict(loaded[\'model\'])\n    clone_opt = torch.optim.AdamW(clone.parameters(), lr=lr, weight_decay=cfg[\'weight_decay\'])\n    clone_opt.load_state_dict(loaded[\'optimizer\'])\n    clone_gen = torch.Generator()\n    clone_gen.set_state(loaded[\'sampler_rng\'].cpu())\n    next_a = torch.randint(len(splits[\'train\']) - cfg[\'context\'], (cfg[\'batch_size\'],), generator=gen)\n    next_b = torch.randint(len(splits[\'train\']) - cfg[\'context\'], (cfg[\'batch_size\'],), generator=clone_gen)\n    assert torch.equal(next_a, next_b)\n    # Work on a copy so verification cannot alter the measured final model.\n    reference = copy.deepcopy(model)\n    ref_opt = torch.optim.AdamW(reference.parameters(), lr=lr, weight_decay=cfg[\'weight_decay\'])\n    ref_opt.load_state_dict(copy.deepcopy(opt.state_dict()))\n    update(reference, ref_opt, windows(splits[\'train\'], next_a, cfg[\'context\']), device, cfg)\n    update(clone, clone_opt, windows(splits[\'train\'], next_b, cfg[\'context\']), device, cfg)\n    resume_error = max((x-y).abs().max().item() for x,y in zip(reference.parameters(), clone.parameters()))\n    assert resume_error < 1e-6, resume_error\n    return records, {\'seconds_including_eval_and_save_excluded\': elapsed,\n                     \'target_tokens_per_second_including_eval\': steps * cfg[\'batch_size\'] / elapsed,\n                     \'resume_next_step_max_parameter_error\': resume_error}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--config\', default=\'configs/routing.json\')\n    parser.add_argument(\'--data\', default=\'data/tinyshakespeare.txt\')\n    parser.add_argument(\'--device\', default=\'cpu\')\n    parser.add_argument(\'--output\', default=\'results/routing\')\n    parser.add_argument(\'--seeds\', type=int, nargs=\'+\')\n    parser.add_argument(\'--threads\', type=int, default=2)\n    args = parser.parse_args()\n    torch.set_num_threads(args.threads)\n    cfg = json.loads(Path(args.config).read_text())\n    if args.seeds:\n        cfg[\'seeds\'] = args.seeds\n    out = Path(args.output)\n    out.mkdir(parents=True, exist_ok=False)  # never overwrite previous evidence\n    cfg_path = out / \'config.json\'\n    cfg_path.write_text(json.dumps(cfg, indent=2))\n    splits, manifest = load_data(Path(args.data), cfg[\'context\'])\n    (out / \'data_manifest.json\').write_text(json.dumps(manifest, indent=2))\n    evals = {k: fixed_batch(v, cfg[\'context\'], cfg[\'eval_samples\'], 9000+i) for i,(k,v) in enumerate(splits.items())}\n    try:\n        commit = subprocess.check_output([\'git\',\'rev-parse\',\'HEAD\'], stderr=subprocess.DEVNULL, text=True).strip()\n    except subprocess.CalledProcessError:\n        commit = None\n    source_hashes = {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(Path(\'src\').glob(\'*.py\'))}\n    environment = {\'python\': platform.python_version(), \'torch\': str(torch.__version__),\n                   \'platform\': platform.platform(), \'device\': args.device, \'threads\': args.threads,\n                   \'git_commit\': commit, \'source_sha256\': source_hashes,\n                   \'gpu_name\': torch.cuda.get_device_name() if args.device.startswith(\'cuda\') else None}\n    (out / \'environment.json\').write_text(json.dumps(environment, indent=2))\n    all_records, summaries = [], []\n    for seed in cfg[\'seeds\']:\n        torch.manual_seed(seed)\n        checkpoint_dir = Path(\'checkpoints\') / out.name / str(seed)\n        checkpoint_dir.mkdir(parents=True, exist_ok=True)\n        dense = LinearLM(cfg[\'context\'], len(manifest[\'vocabulary\'])).to(args.device)\n        records, timing = run_phase(dense,cfg,splits,evals,args.device,seed,\'dense_pretrain\',\n                                   cfg[\'dense_steps\'],cfg[\'dense_lr\'],0,checkpoint_dir)\n        all_records += records\n        initial, dense_end = records[0], records[-1]\n        moe = SparseMoE(dense, cfg[\'experts\'], cfg[\'top_k\']).to(args.device)\n        with torch.no_grad():\n            probe = evals[\'validation\'][0].to(args.device)\n            max_error = (dense(probe)[0] - moe(probe)[0]).abs().max().item()\n        assert max_error < 1e-5, max_error\n        torch.save({\'model\':moe.state_dict(),\'config\':cfg,\'seed\':seed},checkpoint_dir/\'moe_converted.pt\')\n        moe_initial_state = copy.deepcopy(moe.state_dict())\n        branches = {}\n        for name, model in [(\'dense_control\', copy.deepcopy(dense)), (\'moe_continue\', moe)]:\n            recs, stats = run_phase(model,cfg,splits,evals,args.device,seed,name,\n                                   cfg[\'continuation_steps\'],cfg[\'continuation_lr\'],cfg[\'dense_steps\'],checkpoint_dir)\n            all_records += recs\n            # Test is measured only at final checkpoints, never used for model selection.\n            branches[name] = {\'start\':recs[0], \'final\':recs[-1], \'test_ce\':evaluate(model,evals[\'test\'],args.device)[\'ce\'], **stats}\n        expert_deltas = [(expert.weight - moe_initial_state[f\'experts.{i}.weight\']).norm().item()\n                         for i,expert in enumerate(moe.experts)]\n        summary = {\'seed\':seed,\'dense_parameters\':parameters(dense),\'moe_total_parameters\':parameters(moe),\n                   \'moe_active_parameters_per_target\':parameters(moe.router)+cfg[\'top_k\']*parameters(dense.linear),\n                   \'conversion_max_logit_error\':max_error,\'initial\':initial,\'dense_end\':dense_end,\n                   \'dense_pretrain_timing\':timing, \'branches\':branches, \'expert_weight_change_l2\':expert_deltas,\n                   \'expert_pair_distance_l2\':(moe.experts[0].weight-moe.experts[1].weight).norm().item()}\n        summary[\'acceptance\'] = {\'dense_train_loss_drops\':dense_end[\'train_ce\'] < initial[\'train_ce\'],\n                                \'dense_validation_loss_drops\':dense_end[\'validation_ce\'] < initial[\'validation_ce\'],\n                                \'moe_train_loss_drops\':branches[\'moe_continue\'][\'final\'][\'train_ce\'] < dense_end[\'train_ce\'],\n                                \'moe_validation_loss_drops\':branches[\'moe_continue\'][\'final\'][\'validation_ce\'] < dense_end[\'validation_ce\'],\n                                \'all_experts_updated\':all(x>0 for x in expert_deltas),\n                                \'conversion_preserved\':max_error<1e-5}\n        summaries.append(summary)\n        (out / \'summary.json\').write_text(json.dumps(summaries, indent=2))\n        with (out / \'metrics.jsonl\').open(\'w\') as f:\n            for rec in all_records:\n                f.write(json.dumps(rec)+\'\\n\')\n    keys = [\'seed\',\'phase\',\'step\',\'global_step\',\'targets_seen\',\'train_ce\',\'validation_ce\',\'elapsed_seconds_including_eval\',\'batch_ce\',\'objective\',\'grad_norm\',\'router_entropy\']\n    with (out / \'metrics.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=keys,extrasaction=\'ignore\'); writer.writeheader(); writer.writerows(all_records)\n    print(\'Acceptance:\', json.dumps([s[\'acceptance\'] for s in summaries]))\n    if not all(all(s[\'acceptance\'].values()) for s in summaries):\n        raise SystemExit(\'One or more acceptance checks failed; preserve evidence and investigate.\')\n\nif __name__ == \'__main__\':\n    main()\n', 'src/models.py': '"""Literal affine predictor and sparse, function-preserving MoE conversion."""\nimport copy\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nclass LinearLM(nn.Module):\n    def __init__(self, context, vocab):\n        super().__init__()\n        self.context, self.vocab = context, vocab\n        self.linear = nn.Linear(context * vocab, vocab)\n\n    def features(self, tokens):\n        return F.one_hot(tokens, self.vocab).flatten(1).float()\n\n    def forward(self, tokens):\n        return self.linear(self.features(tokens)), None\n\nclass SparseMoE(nn.Module):\n    def __init__(self, dense, experts=4, top_k=2):\n        super().__init__()\n        if not 1 <= top_k <= experts:\n            raise ValueError(\'Require 1 <= top_k <= experts\')\n        self.context, self.vocab = dense.context, dense.vocab\n        self.top_k = top_k\n        self.experts = nn.ModuleList([copy.deepcopy(dense.linear) for _ in range(experts)])\n        self.router = nn.Linear(self.context * self.vocab, experts)\n        nn.init.normal_(self.router.weight, std=0.02)\n        nn.init.zeros_(self.router.bias)\n\n    def forward(self, tokens):\n        x = F.one_hot(tokens, self.vocab).flatten(1).float()\n        router_logits = self.router(x)\n        probs = router_logits.softmax(-1)\n        selected_logits, ids = router_logits.topk(self.top_k, dim=-1)\n        weights = selected_logits.softmax(-1)\n        output = x.new_zeros(x.shape[0], self.vocab)\n        # Actual sparse dispatch: each expert sees only its assigned samples.\n        for i, expert in enumerate(self.experts):\n            rows, slots = torch.where(ids == i)\n            if rows.numel():\n                contribution = expert(x[rows]) * weights[rows, slots, None]\n                output = output.index_add(0, rows, contribution)\n        counts = torch.bincount(ids.flatten(), minlength=len(self.experts)).float()\n        fraction = counts / ids.numel()\n        balance = len(self.experts) * (fraction.detach() * probs.mean(0)).sum()\n        entropy = -(probs * probs.clamp_min(1e-9).log()).sum(-1).mean()\n        return output, {\'balance\': balance, \'counts\': counts, \'entropy\': entropy}\n\ndef parameters(model):\n    return sum(p.numel() for p in model.parameters())\n', 'configs/gpt_colab.json': '{\n  "model": {\n    "context": 128,\n    "layers": 4,\n    "width": 128,\n    "heads": 4,\n    "intermediate": 256,\n    "experts": 4,\n    "top_k": 2\n  },\n  "batch_size": 16,\n  "dense_steps": 1000,\n  "continuation_steps": 1000,\n  "dense_lr": 0.0006,\n  "continuation_lr": 0.0003,\n  "weight_decay": 0.01,\n  "gradient_clip": 1.0,\n  "balance_coefficient": 0.01,\n  "eval_every": 100,\n  "eval_sequences": 64,\n  "seeds": [\n    17\n  ]\n}\n', 'scripts/download_data.py': '"""Download and checksum the exact Tiny Shakespeare corpus."""\nimport hashlib\nimport urllib.request\nfrom pathlib import Path\nimport json\nroot=Path(__file__).resolve().parents[1]\nmanifest=json.loads((root/\'data_specs/tinyshakespeare.json\').read_text())\npath=root/\'data/tinyshakespeare.txt\'\npath.parent.mkdir(exist_ok=True)\nraw=path.read_bytes() if path.exists() else urllib.request.urlopen(manifest[\'source\'],timeout=60).read()\nactual=hashlib.sha256(raw).hexdigest()\nif actual != manifest[\'sha256\']:\n    raise SystemExit(f\'Corpus checksum mismatch: {actual}; investigate upstream change before training.\')\npath.write_bytes(raw)\nprint(f\'Verified {len(raw):,} bytes; SHA-256 {actual}\')\n', 'scripts/report_gpt.py': '"""Plot and report a GPT upcycling run without selecting a best checkpoint."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport statistics as st\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\np=argparse.ArgumentParser()\np.add_argument(\'--results\',required=True)\na=p.parse_args()\nroot=Path(a.results)\nrows=[json.loads(line) for line in (root/\'metrics.jsonl\').read_text().splitlines()]\nsummaries=json.loads((root/\'summary.json\').read_text())\nenv=json.loads((root/\'environment.json\').read_text())\nexecution=f"{env[\'device\']} / {env.get(\'gpu_name\') or \'CPU\'} / PyTorch {env[\'torch\']}"\nfig,axes=plt.subplots(1,2,figsize=(12,4),layout=\'constrained\')\ncolors={\'dense_pretrain\':\'#475569\',\'dense_control\':\'#d97706\',\'moe_continue\':\'#087f8c\'}\nfor phase,color in colors.items():\n    r=[v for v in rows if v[\'phase\']==phase]\n    steps=sorted({v[\'global_step\'] for v in r})\n    for ax,metric in zip(axes,[\'train_ce\',\'validation_ce\']):\n        values=[[v[metric] for v in r if v[\'global_step\']==s] for s in steps]\n        ax.plot(steps,[st.mean(v) for v in values],label=phase.replace(\'_\',\' \'),color=color)\n        if len(summaries)>1:\n            ax.fill_between(steps,[min(v) for v in values],[max(v) for v in values],color=color,alpha=.15)\nfor ax,title in zip(axes,[\'Training evaluation loss\',\'Held-out validation loss\']):\n    ax.set(title=title,xlabel=\'Optimizer steps along each branch\',ylabel=\'Cross entropy (nats / character)\')\n    ax.axvline(summaries[0][\'dense_end\'][\'step\'],color=\'#94a3b8\',ls=\'--\')\n    ax.grid(alpha=.2)\n    ax.legend(fontsize=8)\nfig.suptitle(f"Small GPT → FFN MoE | {len(summaries)} seed(s)")\nfig.savefig(root/\'loss_curves.png\',dpi=170)\nfig.savefig(root/\'loss_curves.svg\')\nlines=[\'# Small GPT → MoE results\',\'\',\n\'| Seed | Initial validation | Trained dense GPT | Continued dense | Continued MoE | MoE test |\',\n\'|---|---:|---:|---:|---:|---:|\']\nfor s in summaries:\n b=s[\'branches\']\n lines.append(f"| {s[\'seed\']} | {s[\'initial\'][\'validation_ce\']:.6f} | {s[\'dense_end\'][\'validation_ce\']:.6f} | {b[\'dense_control\'][\'final\'][\'validation_ce\']:.6f} | {b[\'moe_continue\'][\'final\'][\'validation_ce\']:.6f} | {b[\'moe_continue\'][\'test_ce\']:.6f} |")\ns=summaries[0]\nlines += [\'\',f"Dense: **{s[\'dense_parameters\'][\'total\']:,}** parameters; MoE: **{s[\'moe_parameters\'][\'total\']:,}** total, **{s[\'moe_parameters\'][\'active_parameter_convention\']:,}** under the documented active-parameter convention.",\n\'\',f"Architecture: {s[\'model_config\'][\'layers\']} causal Transformer layers, width {s[\'model_config\'][\'width\']}, {s[\'model_config\'][\'heads\']} heads, context {s[\'model_config\'][\'context\']}, SwiGLU FFNs, tied token/output weights and learned positions. Four experts per FFN, top-2 normalized routing, dropless dispatch. No dropout or autocast in this baseline.",\n\'\',f"Maximum conversion logit error across seeds: {max(s[\'conversion_max_logit_error\'] for s in summaries):.9g}.",\n\'\', \'Acceptance per seed:\',\'\']\nfor s in summaries:\n lines.append(f"- Seed {s[\'seed\']}: "+json.dumps(s[\'acceptance\']))\nlines += [\'\', \'## Interpretation\',\'\',\n\'This is a dense GPT-to-MoE continuation experiment. Dense and MoE losses are measured at every next-character position using causal attention. Every expert matrix updated. Saved models, optimizer and sampler states reproduced the next update within 1e-6. No semantic expert specialization is established.\',\n\'\',\n\'Both continuation branches reset AdamW, use the same lower learning rate and identical sampled sequence order. Expert routing alone changes the FFN path; attention, embeddings, norms and output head are copied. Task CE is separate from the mean layer-balancing term. The reported timing includes evaluation and excludes saving/reload checks. There is no learning-rate sweep or best-checkpoint selection.\',\n\'\',\n\'Validation/test use fixed sampled windows from an 80/10/10 raw-text split. Windows cannot cross split boundaries, but windows within a split may overlap. This short-context character model is a mechanism experiment. Its character CE cannot be compared directly to subword-token CE across different prediction objectives.\',\n\'\',\nf\'Measured execution: {execution}. {len(summaries)} seed(s); these results do not establish universal superiority.\',\n\'\',\n\'Top-2 routing increases active FFN work relative to dense; neither total parameters nor FLOPs are matched. Any small loss difference needs repeated seeds and fair compute controls before interpretation.\',\n\'\', \'![Loss curves](loss_curves.png)\',\'\']\n(root/\'REPORT.md\').write_text(\'\\n\'.join(lines))\nprint(\'\\n\'.join(lines[:13]))\n', 'tests/test_gpt.py': "import unittest\nimport torch\nfrom src.gpt import GPTConfig, TinyGPT, ExpertFFN\n\nclass GPTTests(unittest.TestCase):\n    def setUp(self):\n        torch.manual_seed(17)\n        torch.set_num_threads(2)\n        self.model=TinyGPT(GPTConfig(vocab_size=13,context=12,layers=2,width=16,heads=4,intermediate=32))\n        self.tokens=torch.randint(13,(3,12))\n\n    def test_conversion_preserves_full_gpt_logits(self):\n        converted=self.model.upcycle()\n        torch.testing.assert_close(self.model(self.tokens)[0],converted(self.tokens)[0],atol=1e-6,rtol=1e-5)\n        self.assertEqual(len(converted(self.tokens)[1]),2)\n        self.assertIs(converted.head.weight,converted.token_embedding.weight)\n\n    def test_future_token_does_not_change_past_logits(self):\n        for model in (self.model,self.model.upcycle()):\n            changed=self.tokens.clone()\n            changed[:,7:] = (changed[:,7:]+1)%13\n            torch.testing.assert_close(model(self.tokens)[0][:,:7],model(changed)[0][:,:7])\n\n    def test_sparse_ffn_reference_and_gradients(self):\n        moe=self.model.upcycle().blocks[0].ffn\n        with torch.no_grad():\n            moe.experts[0].down.weight.add_(.1)\n        x=torch.randn(3,12,16,requires_grad=True)\n        flat=x.reshape(-1,16)\n        values,ids=moe.router(flat).topk(2,dim=-1)\n        all_outputs=torch.stack([e(flat) for e in moe.experts],dim=1)\n        ref=(all_outputs[torch.arange(len(flat))[:,None],ids]*values.softmax(-1)[:,:,None]).sum(1)\n        actual,aux=moe(x)\n        torch.testing.assert_close(actual.reshape(-1,16),ref)\n        self.assertEqual(int(aux['counts'].sum()),2*len(flat))\n        actual.square().mean().backward()\n        self.assertGreater(moe.router.weight.grad.norm().item(),0)\n\n    def test_parameter_accounting(self):\n        counts=self.model.upcycle().parameter_counts()\n        self.assertGreater(counts['total'],counts['active_parameter_convention'])\n        self.assertGreater(counts['active_parameter_convention'],self.model.parameter_counts()['total'])\n\nif __name__=='__main__':\n    unittest.main()\n", 'tests/test_models.py': "import unittest\nimport torch\nfrom src.models import LinearLM, SparseMoE\n\nclass ConversionTests(unittest.TestCase):\n    def setUp(self):\n        torch.manual_seed(5)\n        torch.set_num_threads(2)\n        self.dense = LinearLM(4, 7)\n        self.x = torch.randint(7, (31, 4))\n\n    def test_conversion_for_different_k(self):\n        for k in (1, 2, 4):\n            moe = SparseMoE(self.dense, 4, k)\n            torch.testing.assert_close(moe(self.x)[0], self.dense(self.x)[0])\n\n    def test_sparse_dispatch_matches_dense_reference(self):\n        moe = SparseMoE(self.dense, 4, 2)\n        with torch.no_grad():\n            for expert in moe.experts:\n                expert.weight.add_(torch.randn_like(expert.weight)*.1)\n        x = self.dense.features(self.x)\n        values, ids = moe.router(x).topk(2, dim=-1)\n        weights = values.softmax(-1)\n        all_outputs = torch.stack([e(x) for e in moe.experts], dim=1)\n        expected = (all_outputs[torch.arange(len(x))[:,None], ids] * weights[:,:,None]).sum(1)\n        actual, aux = moe(self.x)\n        torch.testing.assert_close(actual, expected)\n        self.assertEqual(int(aux['counts'].sum()), 2*len(x))\n        actual.square().mean().backward()\n        self.assertGreater(moe.router.weight.grad.norm().item(), 0)\n\n    def test_identical_experts_initial_router_task_gradient(self):\n        moe = SparseMoE(self.dense, 4, 2)\n        moe(self.x)[0].square().mean().backward()\n        self.assertLess(moe.router.weight.grad.abs().max().item(), 1e-6)\n\n    def test_invalid_k(self):\n        with self.assertRaises(ValueError):\n            SparseMoE(self.dense, 4, 5)\n\nif __name__ == '__main__':\n    unittest.main()\n", 'scripts/run_logged.py': '"""Stream child output to notebook and disk, preserving its actual failure."""\nfrom collections import deque\nfrom pathlib import Path\nimport subprocess\n\n\ndef run_logged(command, log_path, cwd=None):\n    """Run a command with combined stdout/stderr; refuse to replace its log."""\n    log_path = Path(log_path)\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    tail = deque(maxlen=40)\n    with log_path.open(\'x\', encoding=\'utf-8\') as log:\n        heading = f\'Command: {command!r}\\n\'\n        print(heading, end=\'\', flush=True)\n        log.write(heading)\n        log.flush()\n        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,\n                              stderr=subprocess.STDOUT, text=True,\n                              encoding=\'utf-8\', errors=\'replace\', bufsize=1) as process:\n            for line in process.stdout:\n                print(line, end=\'\', flush=True)\n                log.write(line)\n                log.flush()\n                tail.append(line)\n            code = process.wait()\n        log.write(f\'\\nExit status: {code}\\n\')\n        log.flush()\n    if code:\n        details = \'\'.join(tail)[-20000:]\n        raise RuntimeError(\n            f\'Command failed with exit status {code}. Full output: {log_path.resolve()}\\n\'\n            f\'Last output lines (the underlying error is here):\\n{details}\'\n        )\n    return log_path\n', 'tests/test_run_logged.py': 'import contextlib\nimport io\nfrom pathlib import Path\nimport sys\nimport tempfile\nimport unittest\nfrom scripts.run_logged import run_logged\n\n\nclass LoggedCommandTests(unittest.TestCase):\n    def test_failure_surfaces_stderr_and_preserves_partial_output(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            path=Path(tmp)/\'failure.log\'\n            with contextlib.redirect_stdout(io.StringIO()):\n                with self.assertRaisesRegex(RuntimeError, \'deliberate training failure\'):\n                    run_logged([sys.executable, \'-u\', \'-c\',\n                        "import sys; print(\'step 1 finished\'); print(\'RuntimeError: deliberate training failure\', file=sys.stderr); sys.exit(3)"], path)\n            text=path.read_text()\n            self.assertIn(\'step 1 finished\',text)\n            self.assertIn(\'RuntimeError: deliberate training failure\',text)\n            self.assertIn(\'Exit status: 3\',text)\n\n    def test_success_and_log_overwrite_refused(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            path=Path(tmp)/\'success.log\'\n            with contextlib.redirect_stdout(io.StringIO()):\n                run_logged([sys.executable, \'-u\', \'-c\', "print(\'finished\')"], path)\n                with self.assertRaises(FileExistsError):\n                    run_logged([sys.executable, \'-c\', "raise Exception(\'must not run\')"], path)\n            self.assertIn(\'finished\',path.read_text())\n            self.assertNotIn(\'must not run\',path.read_text())\n', 'src/conversion_checks.py': '"""Scale-aware FP32 conversion checks with retained behavioral diagnostics."""\nimport copy\nimport torch\nfrom torch.nn import functional as F\n\n\n@torch.no_grad()\ndef conversion_diagnostics(reference, actual, targets, atol=1e-5, rtol=1e-5):\n    if reference.shape != actual.shape:\n        raise ValueError(\'Conversion changed output shape\')\n    difference = (actual-reference).abs()\n    allowed = atol + rtol*reference.abs()\n    finite = bool(torch.isfinite(reference).all() and torch.isfinite(actual).all())\n    dense_ce = F.cross_entropy(reference.reshape(-1,reference.shape[-1]),targets.reshape(-1)).item()\n    moe_ce = F.cross_entropy(actual.reshape(-1,actual.shape[-1]),targets.reshape(-1)).item()\n    close = bool(torch.all(difference <= allowed)) if finite else False\n    ce_close = abs(dense_ce-moe_ce) <= 1e-5\n    return {\'atol\':atol,\'rtol\':rtol,\'ce_atol\':1e-5,\'finite\':finite,\n            \'max_absolute_logit_error\':difference.max().item(),\n            \'rms_logit_error\':difference.square().mean().sqrt().item(),\n            \'max_error_over_allowed\':(difference/allowed).max().item(),\n            \'logits_outside_tolerance\':int((difference>allowed).sum().item()),\n            \'dense_probe_ce\':dense_ce,\'moe_probe_ce\':moe_ce,\'probe_ce_change\':moe_ce-dense_ce,\n            \'max_probability_error\':(actual.softmax(-1)-reference.softmax(-1)).abs().max().item(),\n            \'logits_close\':close,\'ce_close\':ce_close,\'passed\':finite and close and ce_close}\n\n\n@torch.no_grad()\ndef verify_model_conversion(dense, moe, tokens, targets):\n    """Validate full-copy conversion in FP32 and independent FP64 CPU probes.\n\n    Retain the original per-logit FP32 check. If it fails, accept only when\n    a tighter CPU-double reference passes AND explicit FP32 behavioral and\n    error bounds pass. This is not a generic license to ignore mismatches.\n    """\n    stats = conversion_diagnostics(dense(tokens)[0], moe(tokens)[0], targets)\n    indices = torch.linspace(0, len(tokens)-1, min(8, len(tokens))).long()\n    probe = tokens.detach().cpu()[indices]\n    labels = targets.detach().cpu()[indices]\n    dense64 = copy.deepcopy(dense).cpu().double().eval()\n    moe64 = copy.deepcopy(moe).cpu().double().eval()\n    ref = dense64(probe)[0]\n    actual = moe64(probe)[0]\n    difference = (actual-ref).abs()\n    finite64 = bool(torch.isfinite(ref).all() and torch.isfinite(actual).all())\n    close64 = bool(torch.all(difference <= 1e-10 + 1e-10*ref.abs())) if finite64 else False\n    ce_ref = F.cross_entropy(ref.reshape(-1,ref.shape[-1]),labels.reshape(-1)).item()\n    ce_actual = F.cross_entropy(actual.reshape(-1,actual.shape[-1]),labels.reshape(-1)).item()\n    reference_passed = finite64 and close64 and abs(ce_actual-ce_ref) <= 1e-10\n    stats[\'fp64_cpu_reference\'] = {\n        \'sequence_indices\':indices.tolist(),\'atol\':1e-10,\'rtol\':1e-10,\'ce_atol\':1e-10,\n        \'max_absolute_logit_error\':difference.max().item(),\n        \'probe_ce_change\':ce_actual-ce_ref,\'passed\':reference_passed}\n    bounds = {\'max_absolute_logit_error\':1e-4, \'rms_logit_error\':1e-5,\n              \'max_probability_error\':1e-5, \'absolute_probe_ce_change\':1e-5}\n    bounded = (stats[\'finite\'] and stats[\'max_absolute_logit_error\'] <= bounds[\'max_absolute_logit_error\']\n               and stats[\'rms_logit_error\'] <= bounds[\'rms_logit_error\']\n               and stats[\'max_probability_error\'] <= bounds[\'max_probability_error\']\n               and abs(stats[\'probe_ce_change\']) <= bounds[\'absolute_probe_ce_change\'])\n    original_passed = stats[\'passed\']\n    stats[\'fp32_behavior_bounds\'] = bounds\n    stats[\'fp32_behavior_bounds_passed\'] = bounded\n    stats[\'passed\'] = reference_passed and (original_passed or bounded)\n    stats[\'acceptance_method\'] = (\'fp32_per_logit_and_fp64_reference\' if original_passed else\n                                  \'fp64_reference_and_bounded_fp32\') if stats[\'passed\'] else \'rejected\'\n    return stats\n', 'tests/test_conversion_checks.py': "import unittest\nimport torch\nfrom src.conversion_checks import conversion_diagnostics\n\nclass ConversionChecksTests(unittest.TestCase):\n    def test_scaled_roundoff_passes_without_global_absolute_looseness(self):\n        reference=torch.tensor([[[8.,2.,0.]]])\n        actual=reference.clone();actual[0,0,0]+=2.24e-5\n        self.assertTrue(conversion_diagnostics(reference,actual,torch.tensor([[0]]))['passed'])\n        actual=reference.clone();actual[0,0,2]+=2.24e-5\n        self.assertFalse(conversion_diagnostics(reference,actual,torch.tensor([[0]]))['passed'])\n\n    def test_material_change_and_nonfinite_are_rejected(self):\n        reference=torch.tensor([[[2.,1.,0.]]])\n        actual=reference.clone();actual[0,0,1]+=.1\n        self.assertFalse(conversion_diagnostics(reference,actual,torch.tensor([[0]]))['passed'])\n        actual[0,0,0]=float('nan')\n        self.assertFalse(conversion_diagnostics(reference,actual,torch.tensor([[0]]))['passed'])\n\n\nclass ModelConversionReferenceTests(unittest.TestCase):\n    def test_double_reference_preserves_model_and_rejects_corruption(self):\n        from src.gpt import GPTConfig, TinyGPT\n        from src.conversion_checks import verify_model_conversion\n        torch.manual_seed(17)\n        torch.set_num_threads(2)\n        model=TinyGPT(GPTConfig(vocab_size=13,context=8,layers=2,width=16,heads=4,intermediate=32)).eval()\n        moe=model.upcycle().eval()\n        x=torch.randint(13,(3,8));y=torch.randint(13,(3,8))\n        before=model.head.weight.detach().clone()\n        stats=verify_model_conversion(model,moe,x,y)\n        self.assertTrue(stats['passed'])\n        self.assertTrue(stats['fp64_cpu_reference']['passed'])\n        self.assertLess(stats['fp64_cpu_reference']['max_absolute_logit_error'],1e-10)\n        self.assertEqual(model.head.weight.dtype,torch.float32)\n        torch.testing.assert_close(model.head.weight,before)\n        with torch.no_grad():moe.blocks[0].attn.proj.weight.add_(.1)\n        self.assertFalse(verify_model_conversion(model,moe,x,y)['passed'])\n\n    def test_fp64_gate_does_not_downcast_probabilities(self):\n        from src.gpt import GPTConfig, TinyGPT\n        torch.manual_seed(17)\n        model=TinyGPT(GPTConfig(vocab_size=13,context=8,layers=1,width=16,heads=4,intermediate=32)).double().eval()\n        moe=model.upcycle().double().eval()\n        x=torch.randint(13,(2,8))\n        torch.testing.assert_close(model(x)[0],moe(x)[0],atol=1e-12,rtol=1e-12)\n\n    def test_roundoff_fallback_requires_all_behavioral_bounds(self):\n        from unittest.mock import patch\n        from src.gpt import GPTConfig, TinyGPT\n        from src.conversion_checks import verify_model_conversion, conversion_diagnostics\n        model=TinyGPT(GPTConfig(vocab_size=13,context=8,layers=1,width=16,heads=4,intermediate=32)).eval()\n        moe=model.upcycle().eval()\n        x=torch.randint(13,(2,8));y=torch.randint(13,(2,8))\n        stats=conversion_diagnostics(model(x)[0],moe(x)[0],y)\n        stats.update(passed=False,logits_close=False,logits_outside_tolerance=6,\n                     max_absolute_logit_error=3.33786e-5,rms_logit_error=1.22588e-6,\n                     max_probability_error=3.51667e-6,probe_ce_change=0.)\n        with patch('src.conversion_checks.conversion_diagnostics',return_value=stats.copy()):\n            result=verify_model_conversion(model,moe,x,y)\n        self.assertTrue(result['passed'])\n        self.assertEqual(result['acceptance_method'],'fp64_reference_and_bounded_fp32')\n        stats['max_probability_error']=.01\n        with patch('src.conversion_checks.conversion_diagnostics',return_value=stats.copy()):\n            self.assertFalse(verify_model_conversion(model,moe,x,y)['passed'])\n", 'data_specs/tinyshakespeare.json': '{\n  "source": "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",\n  "sha256": "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed",\n  "bytes": 1115394,\n  "characters": 1115394,\n  "vocabulary": [\n    "\\n",\n    " ",\n    "!",\n    "$",\n    "&",\n    "\'",\n    ",",\n    "-",\n    ".",\n    "3",\n    ":",\n    ";",\n    "?",\n    "A",\n    "B",\n    "C",\n    "D",\n    "E",\n    "F",\n    "G",\n    "H",\n    "I",\n    "J",\n    "K",\n    "L",\n    "M",\n    "N",\n    "O",\n    "P",\n    "Q",\n    "R",\n    "S",\n    "T",\n    "U",\n    "V",\n    "W",\n    "X",\n    "Y",\n    "Z",\n    "a",\n    "b",\n    "c",\n    "d",\n    "e",\n    "f",\n    "g",\n    "h",\n    "i",\n    "j",\n    "k",\n    "l",\n    "m",\n    "n",\n    "o",\n    "p",\n    "q",\n    "r",\n    "s",\n    "t",\n    "u",\n    "v",\n    "w",\n    "x",\n    "y",\n    "z"\n  ],\n  "split_characters": {\n    "train": 892315,\n    "validation": 111539,\n    "test": 111540\n  },\n  "split_method": "contiguous 80/10/10; split before constructing windows",\n  "context": 128,\n  "objective": "Causal next-character prediction at every position; input/target shifted by one."\n}'}
for name, source in payload.items():
    path = root / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")
print("Embedded source prepared in", root)


## Data and correctness checks
The download is about 1.1 MB. A saved SHA-256 verifies the exact corpus. Train, validation and test are split before creating sequences. Vocabulary is learned from training characters.

In [ ]:
subprocess.run([sys.executable, "scripts/download_data.py"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


## Seed-17 experiment
4 layers, width 128, context 128, 4 attention heads and SwiGLU FFNs. The dense model trains for 1,000 steps. Its FFNs are cloned into four experts per layer with top-2 routing. MoE and dense control each continue for 1,000 steps on identical sampled training sequences. The trainer prints CE, load fractions and gradient statistics and saves checkpoints. FP32 is intentional for the first correctness baseline; this is not an optimized MoE kernel benchmark.

In [ ]:
from datetime import datetime, timezone
from scripts.run_logged import run_logged
stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
run_dir = "results/gpt-colab-" + stamp
training_log = "work/logs/gpt-training-" + stamp + ".log"
print("Training output will be saved to:", training_log)
# -u keeps Python output unbuffered; stderr is combined with stdout and saved.
# A failure displays the actual child traceback, not only CalledProcessError.
run_logged([sys.executable, "-u", "-m", "src.train_gpt",
            "--config", "configs/gpt_colab.json", "--device", "cuda", "--output", run_dir], training_log)
run_logged([sys.executable, "-u", "scripts/report_gpt.py", "--results", run_dir],
           "work/logs/gpt-report-" + stamp + ".log")


## If training fails
`CalledProcessError` alone is a wrapper, not a diagnosis. The training cell now streams and saves the child process’s full stdout/stderr and includes its final traceback in the raised error. Preserve that log and any partial metrics/checkpoints.

To inspect a log after an interrupted run, use `print(Path(training_log).read_text()[-20000:])`. Run the final archive/download cell even if training did not complete. Do not adjust model sizes or numeric tolerances until the actual error is known.

Full-copy conversion now records exact expert-copy checks, per-logit `atol=1e-5, rtol=1e-5` diagnostics and a probe-CE-change limit of `1e-5`. This is a scale-aware FP32 check, not exact-bit equality. To reuse a completed dense phase after a prior conversion failure, see the self-contained `scripts/colab_gpt_recover_cell.py` in the project.

The latest validation also independently checks copied models on CPU in FP64 (gate included) at `atol=rtol=1e-10`. FP32 per-logit violations can pass only with this reference check and recorded bounds on maximum/RMS logit errors, probabilities and task CE. This is a separate, explicit acceptance method; failed per-logit statistics remain in the report.


In [ ]:
from IPython.display import display, Image, Markdown
display(Markdown(Path(run_dir, "REPORT.md").read_text()))
display(Image(filename=str(Path(run_dir, "loss_curves.png"))))
print("Generation samples (illustrative; not model selection):")
print(Path(run_dir, "generation_samples.json").read_text())


## Save evidence
The archive contains configs, source, tests, measured results, phase checkpoints and `work/logs/` subprocess output. If training fails, run this archive cell anyway to preserve the traceback and partial run before debugging. Do not overwrite the original evidence or silently choose a better seed.

Full-copy conversion now records exact expert-copy checks, per-logit `atol=1e-5, rtol=1e-5` diagnostics and a probe-CE-change limit of `1e-5`. This is a scale-aware FP32 check, not exact-bit equality. To reuse a completed dense phase after a prior conversion failure, see the self-contained `scripts/colab_gpt_recover_cell.py` in the project.

The latest validation also independently checks copied models on CPU in FP64 (gate included) at `atol=rtol=1e-10`. FP32 per-logit violations can pass only with this reference check and recorded bounds on maximum/RMS logit errors, probabilities and task CE. This is a separate, explicit acceptance method; failed per-logit statistics remain in the report.


In [ ]:
import shutil
archive = shutil.make_archive("/content/MoE_Training-results", "zip", root_dir=root)
from google.colab import files
files.download(archive)
